# 01 - Tratamento e preparação da base

Tech Challenge Fase 1 - Case NPS Preditivo

Neste notebook eu faço a parte de entendimento e preparação dos dados do CRISP-DM. A ideia é conhecer a base antes de qualquer gráfico mais elaborado: ver o que cada linha representa, os tipos das variáveis, se tem faltante, duplicado ou valor estranho, e decidir o que fazer com cada problema.

No final salvo uma versão tratada em `data/processed/`, que é a que a EDA e o modelo vão usar. A base original em `data/raw/` não é alterada.

Resumo do que vai acontecer aqui:

1. Carregar a base e entender a estrutura
2. Checar qualidade (faltantes, duplicados, faixas de valores)
3. Investigar as inconsistências que apareceram na primeira olhada
4. Tratar a nota de NPS (casas decimais) e criar a classificação
5. Marcar as colunas com risco de vazamento de dados
6. Criar algumas variáveis de apoio para a EDA e salvar

In [25]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# deixa a pasta src visível para importar as funções do projeto
PASTA_PROJETO = Path.cwd().parent
sys.path.append(str(PASTA_PROJETO / "src"))

from nps_utils import classificar_nps, calcular_nps, resumo_por_grupo

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

CAMINHO_BRUTO = PASTA_PROJETO / "data" / "raw" / "desafio_nps_fase_1.csv"
CAMINHO_TRATADO = PASTA_PROJETO / "data" / "processed" / "nps_tratado.csv"

## 1. Estrutura da base

In [26]:
df_bruto = pd.read_csv(CAMINHO_BRUTO)
print(f"Linhas: {df_bruto.shape[0]} | Colunas: {df_bruto.shape[1]}")
df_bruto.head()

Linhas: 2500 | Colunas: 19


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.90,0,3,6.50
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.40,0,3,0.00
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.80,0,7,1.50
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.90,0,4,0.30
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.10,0,3,7.90


In [27]:
df_bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                2500 non-null   int64  
 1   customer_age               2500 non-null   int64  
 2   customer_region            2500 non-null   str    
 3   customer_tenure_months     2500 non-null   int64  
 4   order_id                   2500 non-null   int64  
 5   order_value                2500 non-null   float64
 6   items_quantity             2500 non-null   int64  
 7   discount_value             2500 non-null   float64
 8   payment_installments       2500 non-null   int64  
 9   delivery_time_days         2500 non-null   int64  
 10  delivery_delay_days        2500 non-null   int64  
 11  freight_value              2500 non-null   float64
 12  delivery_attempts          2500 non-null   int64  
 13  customer_service_contacts  2500 non-null   int64  
 14  res

Cada linha é um pedido. Conferindo os identificadores abaixo, cada cliente aparece uma vez só e cada pedido também, então nesta base pedido e cliente são a mesma unidade de análise. Isso simplifica bastante (não preciso agregar nada), mas também quer dizer que não dá para olhar histórico de um mesmo cliente.

In [28]:
print("Clientes únicos:", df_bruto["customer_id"].nunique())
print("Pedidos únicos: ", df_bruto["order_id"].nunique())

Clientes únicos: 2500
Pedidos únicos:  2500


## 2. Qualidade dos dados

### 2.1 Faltantes e duplicados

In [29]:
faltantes = df_bruto.isna().sum()
print("Total de valores faltantes:", faltantes.sum())
print("Linhas duplicadas:", df_bruto.duplicated().sum())
print("Linhas duplicadas ignorando os ids:", df_bruto.drop(columns=["customer_id", "order_id"]).duplicated().sum())

Total de valores faltantes: 0
Linhas duplicadas: 0
Linhas duplicadas ignorando os ids: 0


Nenhum faltante e nenhuma linha repetida, nem quando tiro os ids da comparação. Base bem limpa nesse sentido.

### 2.2 Faixas de valores

Aqui o objetivo é ver se tem valor impossível: idade negativa, nota fora de 0 a 10 e atraso negativo

In [30]:
df_bruto.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,"2,500.00","1,250.50",721.83,1.00,625.75,"1,250.50","1,875.25","2,500.00"
customer_age,"2,500.00",43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,"2,500.00",61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_id,"2,500.00","51,250.50",721.83,"50,001.00","50,625.75","51,250.50","51,875.25","52,500.00"
order_value,"2,500.00",434.26,289.77,7.76,220.25,375.51,577.29,"1,983.81"
items_quantity,"2,500.00",3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,"2,500.00",29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,"2,500.00",6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,"2,500.00",8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,"2,500.00",2.19,1.45,0.00,1.00,2.00,3.00,8.00


In [31]:
regras_faixa = {
    "customer_age": (18, 100),
    "nps_score": (0, 10),
    "csat_internal_score": (0, 10),
    "repeat_purchase_30d": (0, 1),
}
for coluna, (minimo, maximo) in regras_faixa.items():
    fora = (~df_bruto[coluna].between(minimo, maximo)).sum()
    print(f"{coluna:22s} fora de [{minimo}, {maximo}]: {fora}")

colunas_numericas = df_bruto.select_dtypes("number").columns
negativos = (df_bruto[colunas_numericas] < 0).sum()
print("\nColunas com valor negativo:", list(negativos[negativos > 0].index))

customer_age           fora de [18, 100]: 0
nps_score              fora de [0, 10]: 0
csat_internal_score    fora de [0, 10]: 0
repeat_purchase_30d    fora de [0, 1]: 0

Colunas com valor negativo: []


In [32]:
df_bruto["customer_region"].value_counts()

customer_region
Sul             521
Sudeste         520
Norte           506
Nordeste        485
Centro-Oeste    468
Name: count, dtype: int64

Nada fora de faixa e nenhum valor negativo. As cinco regiões têm quantidades parecidas, o que é bom para comparar regiões depois.

Uma coisa que chamou atenção no `describe`: a média de 4 reclamações por pedido (`complaints_count`). Para uma única compra parece muito. Pode ser que a coluna conte o histórico do cliente e não só esse pedido. 

## 3. Inconsistências

### 3.1 Atraso maior que o tempo total de entrega

Pela definição do dicionário, `delivery_time_days` é o tempo total de entrega e `delivery_delay_days` é quantos dias atrasou. Logicamente o atraso deveria ser menor ou igual ao tempo total (um pedido que levou 2 dias não pode ter atrasado 4).

In [33]:
mascara_atraso = df_bruto["delivery_delay_days"] > df_bruto["delivery_time_days"]
print(f"Pedidos com atraso maior que o tempo total: {mascara_atraso.sum()} ({mascara_atraso.mean():.1%})")

pd.crosstab(
    df_bruto.loc[mascara_atraso, "delivery_time_days"],
    df_bruto.loc[mascara_atraso, "delivery_delay_days"],
)

Pedidos com atraso maior que o tempo total: 121 (4.8%)


delivery_delay_days,3,4,5,6,7,8
delivery_time_days,,,,,,
2,34,23,8,4,1,0
3,0,20,11,3,2,0
4,0,0,7,3,2,1
5,0,0,0,1,1,0


São 121 pedidos (4,8%), todos com tempo de entrega curto (2 a 4 dias). Pensei em três explicações:

- `delivery_time_days` na verdade é o prazo prometido, e não o tempo real. Nesse caso não teria inconsistência nenhuma.
- Erro de registro em uma das duas colunas.
- A base foi gerada sorteando as colunas de forma independente. A correlação entre as duas é praticamente zero, o que reforça essa hipótese.

Decisão: não excluir. O atraso é justamente a variável que mais parece pesar na nota, e jogar fora 5% da base por uma suspeita que eu não consigo confirmar pode enviesar a análise. Crio uma flag (`flag_atraso_maior_que_entrega`) e, na EDA, refaço as conclusões principais sem esses pedidos para ver se algo muda.

In [34]:
print("Correlação de Spearman entre tempo de entrega e atraso:",
      round(df_bruto[["delivery_time_days", "delivery_delay_days"]].corr(method="spearman").iloc[0, 1], 3))

Correlação de Spearman entre tempo de entrega e atraso: -0.006


### 3.2 Desconto maior que o valor do pedido

In [35]:
mascara_desconto = df_bruto["discount_value"] > df_bruto["order_value"]
print(f"Pedidos com desconto maior que o valor: {mascara_desconto.sum()} ({mascara_desconto.mean():.1%})")
df_bruto.loc[mascara_desconto, ["order_value", "discount_value", "freight_value"]].head(10)

Pedidos com desconto maior que o valor: 35 (1.4%)


,order_value,discount_value,freight_value
6,41.29,99.62,35.83
57,86.15,112.42,36.23
116,60.96,95.06,30.19
182,24.00,85.49,29.58
270,79.93,129.11,27.59
273,47.07,50.24,32.58
297,59.81,64.05,34.68
318,27.57,44.05,26.09
349,22.06,41.75,31.92
359,28.27,30.59,37.13


São 35 pedidos (1,4%), quase todos de valor baixo. Se `order_value` for o valor já com o desconto aplicado, um pedido de R$ 41 com R$ 99 de desconto é possível (o preço cheio seria R$ 140). O dicionário diz apenas "valor total do pedido", então fica ambíguo.

Decisão: mesma lógica do atraso, mantenho e marco com `flag_desconto_maior_que_valor`. Como desconto não parece ter muita relação com o NPS, o impacto dessa escolha deve ser pequeno.

### 3.3 Outliers

Usei a regra do intervalo interquartil (IQR) para contar valores muito acima ou abaixo do padrão nas variáveis contínuas e nas contagens.

In [36]:
def contar_outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({
        "limite_inferior": limite_inf,
        "limite_superior": limite_sup,
        "qtd_outliers": int(((serie < limite_inf) | (serie > limite_sup)).sum()),
        "valor_max": serie.max(),
    })

colunas_outlier = ["order_value", "discount_value", "freight_value", "delivery_delay_days",
                   "customer_service_contacts", "complaints_count", "resolution_time_days"]
df_bruto[colunas_outlier].apply(contar_outliers_iqr).T.astype({"qtd_outliers": int})

,limite_inferior,limite_superior,qtd_outliers,valor_max
order_value,-315.32,"1,112.86",84,"1,983.81"
discount_value,-39.04,88.75,125,230.33
freight_value,5.41,70.78,12,76.13
delivery_delay_days,-2.00,6.00,17,8.00
customer_service_contacts,-0.50,3.50,176,7.00
complaints_count,0.00,8.00,29,11.00
resolution_time_days,-7.00,17.00,0,11.00


Os outliers aparecem principalmente em valor do pedido, desconto e nas contagens de atraso, contato e reclamação. Olhando os valores máximos, nenhum é absurdo para um e-commerce (um pedido de quase R$ 2.000, 8 dias de atraso, 7 contatos com o SAC). São casos extremos mas plausíveis, e alguns deles são exatamente os clientes que mais interessam para o negócio (quem teve a pior experiência).

## 4. Tratamento da nota de NPS

### 4.1 Notas com casas decimais

In [37]:
tem_decimal = (df_bruto["nps_score"] % 1) != 0
print(f"Notas com casas decimais: {tem_decimal.mean():.1%}")
print("Notas exatamente 0: ", (df_bruto["nps_score"] == 0).sum())
print("Notas exatamente 10:", (df_bruto["nps_score"] == 10).sum())
df_bruto["nps_score"].sample(10, random_state=1).tolist()

Notas com casas decimais: 83.0%
Notas exatamente 0:  158
Notas exatamente 10: 42


[4.2, 0.6, 2.1, 0.0, 6.3, 5.2, 5.6, 8.3, 10.0, 5.0]

Numa pesquisa de NPS real o cliente escolhe um número inteiro de 0 a 10. Aqui 83% das notas têm decimal, e tem um acúmulo de 158 notas exatamente iguais a 0.

Com nota decimal surge a dúvida de como classificar valores como 6,5 ou 8,7. A primeira ideia foi arredondar para o inteiro mais próximo, mas desisti: um cliente que deu 6,5 ainda não chegou no 7, que é onde começa a faixa de neutro. Arredondar seria dar meio ponto para ele.

Por isso a classificação é feita direto na nota original, sem arredondamento:

| Nota original | Categoria |
|---|---|
| menor que 7,0 | Detrator |
| de 7,0 até menos de 9,0 | Neutro |
| 9,0 ou mais | Promotor |

A regra fica na função `classificar_nps` em `src/nps_utils.py`, para todos os notebooks usarem exatamente a mesma. Só para registrar o tamanho da decisão, quantos clientes estão na zona que mudaria de categoria se eu tivesse arredondado:

In [38]:
zona_6_5_a_7 = df_bruto["nps_score"].between(6.5, 7.0, inclusive="left")
zona_8_5_a_9 = df_bruto["nps_score"].between(8.5, 9.0, inclusive="left")
print("Notas de 6,5 até menos de 7,0 (detratores pela regra, virariam neutros arredondando):", zona_6_5_a_7.sum())
print("Notas de 8,5 até menos de 9,0 (neutros pela regra, virariam promotores arredondando):", zona_8_5_a_9.sum())

Notas de 6,5 até menos de 7,0 (detratores pela regra, virariam neutros arredondando): 134
Notas de 8,5 até menos de 9,0 (neutros pela regra, virariam promotores arredondando): 46


São 180 pedidos (mais de 7% da base), o que mexe bastante no NPS final. Mais um motivo para deixar a regra escrita e documentada.

### 4.2 Criando a target

In [39]:
df = df_bruto.copy()

df["categoria_nps"] = classificar_nps(df["nps_score"])
# alvo binário que vai ser usado no modelo: 1 = detrator
df["flag_detrator"] = (df["categoria_nps"] == "Detrator").astype(int)

distribuicao = df["categoria_nps"].value_counts(normalize=True).sort_index() * 100
print(distribuicao.round(1))
print(f"\nNPS da base: {calcular_nps(df['categoria_nps']):.1f}")

categoria_nps
Detrator   84.40
Neutro     11.20
Promotor    4.40
Name: proportion, dtype: float64

NPS da base: -80.0


84% de detratores e NPS de -80,0. As classes são bem desbalanceadas, o que vai importar na hora do modelo.

## 5. Vazamento de dados (data leakage)

Duas colunas não podem ser usadas como explicação nem como entrada do modelo, porque são medidas no momento da pesquisa ou depois dela:

- `repeat_purchase_30d`: acontece depois da pesquisa. E tem um problema pior, conferido abaixo.
- `csat_internal_score`: outra nota de satisfação, coletada junto ou depois do NPS.

In [40]:
regra_nota_8 = (df["nps_score"] >= 8.0).astype(int)
print("repeat_purchase_30d é igual a (nps_score >= 8)?", (regra_nota_8 == df["repeat_purchase_30d"]).all())

pd.crosstab(df["categoria_nps"], df["repeat_purchase_30d"], margins=True)

repeat_purchase_30d é igual a (nps_score >= 8)? True


repeat_purchase_30d,0,1,All
categoria_nps,,,
Detrator,2109,0,2109
Neutro,173,108,281
Promotor,0,110,110
All,2282,218,2500


A recompra é exatamente "nota maior ou igual a 8" nos 2.500 pedidos. Ou seja, a coluna foi criada a partir da própria target. Se ela entrar em qualquer modelo, ele vai acertar quase tudo sem aprender nada de útil. Fica marcada como vazamento.

Sobre `complaints_count`, ainda não decidi. Se a contagem for do histórico do cliente ou incluir reclamações feitas depois da pesquisa, também seria vazamento. Isso é investigado na EDA.

In [41]:
COLUNAS_VAZAMENTO = ["repeat_purchase_30d", "csat_internal_score"]
COLUNAS_EM_AVALIACAO = ["complaints_count"]
print("Fora da análise causal e do modelo:", COLUNAS_VAZAMENTO)
print("Em avaliação:", COLUNAS_EM_AVALIACAO)

Fora da análise causal e do modelo: ['repeat_purchase_30d', 'csat_internal_score']
Em avaliação: ['complaints_count']


## 6. Variáveis de apoio para a EDA

Criei algumas faixas para facilitar a leitura dos gráficos por gestores (é mais fácil falar "clientes com 3 ou mais dias de atraso" do que mostrar uma correlação) e duas variáveis derivadas que fazem sentido de negócio:

- `entregue_no_prazo`: 1 se o atraso foi zero.
- `pct_frete_pedido`: quanto o frete representa do valor do pedido. A hipótese é que o cliente sente mais um frete caro em relação ao que comprou do que o valor absoluto.

In [42]:
df["flag_atraso_maior_que_entrega"] = mascara_atraso.astype(int)
df["flag_desconto_maior_que_valor"] = mascara_desconto.astype(int)

df["entregue_no_prazo"] = (df["delivery_delay_days"] == 0).astype(int)
df["pct_frete_pedido"] = df["freight_value"] / df["order_value"] * 100

df["faixa_atraso"] = pd.cut(df["delivery_delay_days"], bins=[-1, 0, 1, 2, 3, 100],
                            labels=["No prazo", "1 dia", "2 dias", "3 dias", "4+ dias"])
df["faixa_contatos"] = pd.cut(df["customer_service_contacts"], bins=[-1, 0, 1, 2, 100],
                              labels=["0", "1", "2", "3+"])
df["faixa_resolucao"] = pd.cut(df["resolution_time_days"], bins=[-1, 2, 5, 8, 100],
                               labels=["0-2 dias", "3-5 dias", "6-8 dias", "9+ dias"])
df["faixa_idade"] = pd.cut(df["customer_age"], bins=[17, 25, 35, 45, 55, 100],
                           labels=["18-25", "26-35", "36-45", "46-55", "56+"])
df["faixa_tempo_casa"] = pd.cut(df["customer_tenure_months"], bins=[0, 12, 36, 60, 1000],
                                labels=["até 1 ano", "1-3 anos", "3-5 anos", "5+ anos"])
df["faixa_valor_pedido"] = pd.qcut(df["order_value"], q=4,
                                   labels=["Q1 (menor)", "Q2", "Q3", "Q4 (maior)"])

# conferindo se nenhuma faixa ficou vazia ou pequena demais
for coluna in ["faixa_atraso", "faixa_contatos", "faixa_resolucao", "faixa_idade", "faixa_tempo_casa", "faixa_valor_pedido"]:
    print(coluna, df[coluna].value_counts().sort_index().to_dict())

faixa_atraso {'No prazo': 277, '1 dia': 615, '2 dias': 646, '3 dias': 525, '4+ dias': 437}
faixa_contatos {'0': 554, '1': 816, '2': 640, '3+': 490}
faixa_resolucao {'0-2 dias': 629, '3-5 dias': 619, '6-8 dias': 637, '9+ dias': 615}
faixa_idade {'18-25': 389, '26-35': 465, '36-45': 517, '46-55': 468, '56+': 661}
faixa_tempo_casa {'até 1 ano': 242, '1-3 anos': 490, '3-5 anos': 478, '5+ anos': 1290}
faixa_valor_pedido {'Q1 (menor)': 625, 'Q2': 625, 'Q3': 625, 'Q4 (maior)': 625}


Uma primeira olhada no NPS por faixa de atraso, só para ver se as faixas fazem sentido antes de salvar:

In [43]:
resumo_por_grupo(df, "faixa_atraso")

,pedidos,nota_media,pct_detratores,pct_promotores,nps
faixa_atraso,,,,,
No prazo,277,6.86,51.62,18.05,-33.57
1 dia,615,5.55,73.98,7.15,-66.83
2 dias,646,4.58,87.93,2.01,-85.91
3 dias,525,3.44,96.76,0.57,-96.19
4+ dias,437,2.00,99.54,0.00,-99.54


Já dá para ver a direção do que a EDA vai aprofundar.

## 7. Salvando a base tratada

In [44]:
df.to_csv(CAMINHO_TRATADO, index=False)
print("Arquivo salvo em:", CAMINHO_TRATADO.relative_to(PASTA_PROJETO))
print("Formato final:", df.shape)

Arquivo salvo em: data/processed/nps_tratado.csv
Formato final: (2500, 31)


Próximo passo: notebook `02_eda.ipynb`.